<a href="https://colab.research.google.com/github/prannav-vg/VLM_fun/blob/main/Video_Anomalies_VLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U transformers accelerate qwen-vl-utils av

import os, subprocess
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from google.colab import files

model_id = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id)

uploaded = files.upload()
original_path = os.path.abspath(next(iter(uploaded)))

small_path = "/content/video_small.mp4"
subprocess.run([
    "ffmpeg", "-y", "-i", original_path,
    "-vf", "scale=-2:480", "-r", "10",
    "-c:v", "libx264", "-preset", "veryfast", "-an",
    small_path
], check=True, capture_output=True)
print("Converted to 480p:", small_path)

def ask_video(question, path=small_path, fps=1.0):
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "video": path, "max_pixels": 360 * 420, "fps": fps},
            {"type": "text", "text": question},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256)
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0]

print(ask_video("is anything unusal happening in the video?"))

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Saving 12995023-hd_1920_1080_25fps.mp4 to 12995023-hd_1920_1080_25fps.mp4
Converted to 480p: /content/video_small.mp4


qwen-vl-utils using torchcodec to read video.
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


The video shows two boxers in a ring, engaged in a boxing match. There is no indication of anything unusual happening in the video.


In [4]:
import os, zipfile, shutil
from google.colab import files

LINK = "https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/ACSm_o3uoOKYRyTX_D4gAlc/Anomaly-Videos-Part-1.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1"
ZIP_PATH = "/content/Anomaly-Videos-Part-1.zip"

if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 1e9:
    !wget -q --show-progress -O "$ZIP_PATH" "$LINK"

size_gb = os.path.getsize(ZIP_PATH) / 1e9
print(f"Downloaded: {size_gb:.2f} GB")
if size_gb < 1:
    raise RuntimeError("Download too small: Dropbox probably sent a web page instead of the zip.")

DATA_DIR = "/content/ucf_test"
os.makedirs(DATA_DIR, exist_ok=True)
ANNOTATION_FILE = f"{DATA_DIR}/Temporal_Anomaly_Annotation_for_Testing_Videos.txt"

if not os.path.exists(ANNOTATION_FILE):
    print("\nUpload Temporal_Anomaly_Annotation_for_Testing_Videos.txt now:")
    up = files.upload()
    shutil.move(next(iter(up)), ANNOTATION_FILE)

wanted = set()
with open(ANNOTATION_FILE) as f:
    for line in f:
        if line.strip():
            wanted.add(line.split()[0])

MAX_VIDEOS = 6
found = 0
with zipfile.ZipFile(ZIP_PATH) as z:
    members = z.namelist()
    print("\nExample files inside the zip:", [m for m in members if m.endswith(".mp4")][:5])
    for member in members:
        base = os.path.basename(member)
        if base in wanted and found < MAX_VIDEOS:
            with z.open(member) as src, open(os.path.join(DATA_DIR, base), "wb") as dst:
                shutil.copyfileobj(src, dst)
            found += 1
            print("Extracted test video:", base)

print(f"\n{found} test videos ready in {DATA_DIR}")

/content/Anomaly-Vi 100%[===================>]   5.84G  21.1MB/s    in 5m 0s   
Downloaded: 6.27 GB

Upload Temporal_Anomaly_Annotation_for_Testing_Videos.txt now:


Saving Anomaly_Train (1).txt to Anomaly_Train (1).txt

Example files inside the zip: ['Anomaly-Videos-Part-1/Abuse/Abuse041_x264.mp4', 'Anomaly-Videos-Part-1/Abuse/Abuse037_x264.mp4', 'Anomaly-Videos-Part-1/Abuse/Abuse012_x264.mp4', 'Anomaly-Videos-Part-1/Abuse/Abuse001_x264.mp4', 'Anomaly-Videos-Part-1/Abuse/Abuse048_x264.mp4']

0 test videos ready in /content/ucf_test


In [6]:
import os, zipfile, shutil
from google.colab import files

DATA_DIR = "/content/ucf_test"
ZIP_PATH = "/content/Anomaly-Videos-Part-1.zip"
ANNOTATION_FILE = f"{DATA_DIR}/Temporal_Anomaly_Annotation_for_Testing_Videos.txt"
os.makedirs(DATA_DIR, exist_ok=True)

for f in [ANNOTATION_FILE, "/content/Anomaly_Train (1).txt"]:
    if os.path.exists(f):
        os.remove(f)

print("Upload Temporal_Anomaly_Annotation_for_Testing_Videos.txt (NOT Anomaly_Train.txt):")
up = files.upload()
shutil.move(next(iter(up)), ANNOTATION_FILE)

with open(ANNOTATION_FILE) as f:
    lines = [l for l in f if l.strip()]
print("\nFirst 3 lines of the file:")
for l in lines[:3]:
    print("  ", l.strip())
if len(lines[0].split()) < 6:
    raise RuntimeError("This is not the temporal annotation file. Each line should have 6 columns: name, class, and 4 frame numbers.")
print(f"Looks correct: {len(lines)} test videos listed.")

wanted = {l.split()[0] for l in lines}
MAX_VIDEOS = 6
found = 0
with zipfile.ZipFile(ZIP_PATH) as z:
    for member in z.namelist():
        base = os.path.basename(member)
        if base in wanted and found < MAX_VIDEOS:
            with z.open(member) as src, open(os.path.join(DATA_DIR, base), "wb") as dst:
                shutil.copyfileobj(src, dst)
            found += 1
            print("Extracted test video:", base)

print(f"\n{found} test videos ready in {DATA_DIR}")

Upload Temporal_Anomaly_Annotation_for_Testing_Videos.txt (NOT Anomaly_Train.txt):


Saving Temporal_Anomaly_Annotation.txt to Temporal_Anomaly_Annotation (1).txt

First 3 lines of the file:
   Abuse028_x264.mp4  Abuse  165  240  -1  -1
   Abuse030_x264.mp4  Abuse  1275  1360  -1  -1
   Arrest001_x264.mp4  Arrest  1185  1485  -1  -1
Looks correct: 290 test videos listed.
Extracted test video: Abuse028_x264.mp4
Extracted test video: Abuse030_x264.mp4
Extracted test video: Assault011_x264.mp4
Extracted test video: Assault006_x264.mp4
Extracted test video: Assault010_x264.mp4
Extracted test video: Arrest039_x264.mp4

6 test videos ready in /content/ucf_test
